# Bank campaign — comprehensive exploratory analysis

This notebook restores variable-level exploration. It profiles every source column, retains missing/unknown groups, compares target-conditioned distributions, checks outliers, studies selected variable pairs and exports a dynamic cohort dashboard. Original walkthroughs remain under `historical/`. No model accuracy or causal effect is claimed.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
from scripts.analysis import run
report = run(mode=os.environ.get("RUN_MODE", "full"))
display(report)


## 1. Population, target and source quality

No target imputation is performed. Bank records without valid response labels are counted and excluded; credit target-rate denominators remain one current applicant each. Missing covariates and outliers are retained.

In [ ]:
profiles = pd.read_csv("results/all_source_variable_profiles.csv")
display(profiles[["table","variable","dtype","rows","missing","missing_pct","unique_nonmissing","role","unknown_marker_count"]])
display(Image(filename="results/missingness.png"))


## 2. Numeric distributions and outliers

The population charts use fixed full-cohort quantile bins, not equal-width density bins. The downloadable profile retains exact descriptive statistics. Outliers use the 1.5×IQR rule and are flagged, not silently deleted.

In [ ]:
numeric = pd.read_csv("results/numeric_by_target.csv")
display(numeric)
display(Image(filename="results/numeric_distributions.png"))


## 3. Categorical variables and target rates

Every table includes group size, positive/negative counts, rate, lift and Wilson 95% intervals. Rare-group rates are unstable; intervals are not adjusted for confounding or multiple comparisons.

In [ ]:
segments = pd.read_csv("results/target_summary.csv")
display(segments.groupby("variable",sort=False).agg(groups=("group","size"),observations=("observations","sum"),positive=("positive","sum")))
display(Image(filename="results/categorical_target_rates.png"))


## 4. Inspect a variable in detail

Change `FOCUS_VARIABLE` to any value in `segments.variable.unique()`. This preserves all groups, including missing values, instead of showing only one selected chart.

In [ ]:
FOCUS_VARIABLE = 'duration_seconds'
display(segments.loc[segments.variable.eq(FOCUS_VARIABLE)])
display(numeric.loc[numeric.variable.eq(FOCUS_VARIABLE)])


## 5. Associations, bivariate relationships and correlation

Binned Cramér’s V is descriptive and sensitive to binning and sparse groups. Point-biserial correlation summarizes numeric-target association. Pearson/Spearman matrices cover documented continuous fields on pairwise complete observations; they do not imply causation.

In [ ]:
associations = pd.read_csv("results/variable_associations.csv")
display(associations)
display(pd.read_csv("results/correlation_spearman.csv",index_col=0))
display(Image(filename="results/correlations.png"))


## 6. Previous-application context (credit only)

Univariate history tables use previous-application rows. Category-presence target tables deduplicate applicant/category pairs and then attach the current target once per pair. Category groups overlap and must not be summed; these are not loan-level default rates.

In [ ]:
history_table = Path("results/previous_category_presence.csv")
if history_table.exists():
    display(pd.read_csv(history_table))
    display(pd.read_csv("results/previous_univariate.csv"))


## 7. Read the findings and open the dynamic dashboard

The dashboard adds four simultaneous cohort filters, every analytical variable, distribution/rate views, uncertainty, selected variable-pair heatmaps, raw source audits and CSV exports. Correlation and raw-source panels are explicitly full-cohort views. See `DASHBOARD.md` for published and local access.

In [ ]:
findings = Path("results/EDA_FINDINGS.md")
if findings.exists():
    display(Markdown(findings.read_text(encoding="utf-8")))
display(pd.read_json("dashboard/manifest.json",typ="series"))
